In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA

ModuleNotFoundError: No module named 'pyspark'

In [35]:
df = pd.read_csv("dados/dataset.csv")
print(df.head())

   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       149610     False   
2              To Begin Again          57       210826     False   


In [36]:
df.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

In [37]:
df.dtypes

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

In [38]:
df = df.drop(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'track_genre', 'key', 'mode','explicit'], axis=1)

In [ ]:
df.columns #quis remover o boleano pq fiquei com medo de interferir muito na análise, mas se for necessário posso colocar ele de volta

Index(['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature'],
      dtype='object')

key: tonalidade da música (0–11, representando C, C#, D, ... B em notação cromática). Numérica discreta.
mode: maior (1) ou menor (0) — a escala da música. Binária.
valence: medida de positividade/felicidade (0–1). Quanto maior, mais alegre/positiva a música. Contínua.
tempo: velocidade em BPM (batidas por minuto). Ex: 120 BPM = andamento moderado. Contínua.
time_signature: compasso/assinatura de tempo (ex: 4/4, 3/4). Discreta; geralmente 4 é mais comum em pop/rock.

ao meu ver, vale a pena deixar o valence, tempo e time signature talvez

In [40]:
df.dtypes

popularity            int64
duration_ms           int64
danceability        float64
energy              float64
loudness            float64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
dtype: object

In [41]:
print("NaNs por coluna:\n", df.isnull().sum())

NaNs por coluna:
 popularity          0
duration_ms         0
danceability        0
energy              0
loudness            0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64


In [ ]:
spark = SparkSession.builder.appName("PCA").getOrCreate()

In [ ]:
spark_df = spark.createDataFrame(df)

In [ ]:
assembler = VectorAssembler(inputCols=spark_df.columns, outputCol="features")
vetor = assembler.transform(spark_df)

In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True,withStd=True)
scaler_model = scaler.fit(vetor)
df_scaled = scaler_model.transform(vetor)